# 01 — EDA and the CV Decision

The output of this notebook is a **validation decision**, not merely a collection of plots.

1. Examine the target distribution, missingness, and cardinality of every categorical feature.
2. Determine how the test set was constructed: a random split, a future time period, or held-out entities. Compare train and test across the ID, time, and group columns until the split strategy can be described in one sentence.
3. Measure the share of test rows belonging to entities that never appear in train. Record this value as `COLD_FRACTION`.
4. Set `CV_SCHEME` in `src/config.py` and document the justification under **Validation decision** in `experiments.md`. Treat this decision as fixed for all subsequent experiments.
5. Save each figure as it is created using:

   ```python
   savefig(fig, "01_<name>")
   ```

In [ ]:
import re
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config as C
from src.data import TARGET_MAP, load, load_raw, load_sample_submission
from src.utils import savefig

sns.set_theme(style="whitegrid")

train, test = load()
sub = load_sample_submission()
train.head()

## 0. What the config says vs. what the files contain

`data.basic_clean` lower-cases every column name and `data.encode_target` maps
the `"Yes"`/`"No"` label to int8 `1`/`0`, so the frames the rest of the repo sees
use lower-case names and a numeric target. `config.TARGET` is set to match.
Section 1 shows what the raw file actually held.

In [ ]:
TARGET = C.TARGET  # "churn" — lower-cased by basic_clean
ID = C.ID_COL
features = [c for c in test.columns if c != ID]

print(f"train {train.shape}   test {test.shape}   sample_sub {sub.shape}")
print(f"test share of labelled+unlabelled rows: {len(test) / (len(train) + len(test)):.4f}")
print(f"\nfeature columns ({len(features)}):\n{features}")
print(f"\nsample_submission columns: {list(sub.columns)}  (read raw — keeps its own casing)")
train.dtypes.to_frame("dtype")

## 1. Target, missingness, cardinality

In [ ]:
# what the file holds, before data.encode_target touches it
raw_train, _ = load_raw()
print(raw_train["Churn"].value_counts(dropna=False).to_string())
print(f"raw label dtype: {raw_train['Churn'].dtype} — text, not 0/1; mapped by {TARGET_MAP}")

y = train[TARGET]
print(f"\nencoded dtype: {y.dtype}")
print(f"positive rate: {y.mean():.6f}   (imbalance ~{(1 - y.mean()) / y.mean():.2f}:1)")
print(f"sample_submission target: {sub['Churn'].dtype}, values {sub['Churn'].unique()}")

In [ ]:
na = pd.concat([train.isna().sum().rename("train_na"), test.isna().sum().rename("test_na")], axis=1)
print("total missing cells:", int(na.sum().sum()))

# TotalCharges is the classic Telco trap: blank strings for tenure-0 customers,
# which make the column parse as object. Confirm it did not happen here.
for name, df in [("train", train), ("test", test)]:
    coerced = pd.to_numeric(df["totalcharges"], errors="coerce")
    print(
        f"{name}: totalcharges dtype={df['totalcharges'].dtype} "
        f"non-numeric={int(coerced.isna().sum())} "
        f"tenure==0 rows={int((df['tenure'] == 0).sum())}"
    )

In [ ]:
cat_cols = [c for c in features if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in features if pd.api.types.is_numeric_dtype(train[c])]

card = []
for c in cat_cols:
    u_tr, u_te = set(train[c].dropna()), set(test[c].dropna())
    card.append(
        {
            "column": c,
            "n_train": len(u_tr),
            "n_test": len(u_te),
            "unseen_in_train": sorted(u_te - u_tr),
            "levels": sorted(u_tr),
        }
    )
print(f"numeric: {num_cols}")
pd.DataFrame(card).set_index("column")

Every categorical is low-cardinality (2–4 levels) and **no level appears in test
that is absent from train**. There is nothing here that needs target encoding or
rare-level collapsing, and native categorical handling in LightGBM/CatBoost is
enough.

`"No internet service"` / `"No phone service"` appear as third levels on the
add-on columns — they are structurally implied by `internetservice == "No"` and
`phoneservice == "No"`, not independent information.

In [ ]:
# int8 target would give seaborn a continuous hue; label it for the plots only
plot_df = train.assign(churned=train[TARGET].map({0: "No", 1: "Yes"}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x=plot_df["churned"], order=["No", "Yes"], ax=axes[0])
axes[0].set_title(f"Target — positive rate {y.mean():.3f}")
sns.histplot(
    data=plot_df,
    x="tenure",
    hue="churned",
    hue_order=["No", "Yes"],
    bins=72,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1],
)
axes[1].set_title("tenure by churn")
fig.tight_layout()
savefig(fig, "01_target_and_tenure")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(
    axes.ravel(),
    [
        "contract",
        "internetservice",
        "paymentmethod",
        "techsupport",
        "onlinesecurity",
        "seniorcitizen",
    ],
):
    rate = y.groupby(train[col]).mean().sort_values()
    sns.barplot(x=rate.values, y=rate.index.astype(str), ax=ax)
    ax.axvline(y.mean(), color="k", ls="--", lw=1)
    ax.set_xlabel("churn rate")
    ax.set_title(col)
fig.suptitle("Churn rate by category (dashed = base rate)")
fig.tight_layout()
savefig(fig, "01_churn_rate_by_category")

In [ ]:
from sklearn.metrics import roc_auc_score

uni = []
for c in features:
    if pd.api.types.is_numeric_dtype(train[c]):
        s = train[c]
    else:
        s = train[c].map(y.groupby(train[c]).mean())  # rank by in-sample rate
    uni.append((c, max(roc_auc_score(y, s), 1 - roc_auc_score(y, s))))
uni = pd.DataFrame(uni, columns=["feature", "univariate_auc"]).sort_values(
    "univariate_auc", ascending=False
)

fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(data=uni, x="univariate_auc", y="feature", ax=ax)
ax.axvline(0.5, color="k", ls="--", lw=1)
ax.set_title("Univariate AUC (categoricals scored by in-sample rate — optimistic)")
fig.tight_layout()
savefig(fig, "01_univariate_auc")
uni

`tenure`, `contract`, `paymentmethod` and `internetservice` carry most of the
single-feature signal; `gender` and `phoneservice` carry none. The categorical
AUCs are computed on in-sample rates and so are mildly optimistic, but with
2–4 levels each the bias is negligible.

## 2. How was the test set built?

Three candidate mechanisms, each with a measurement that discriminates it:

| Mechanism | What it would look like |
|---|---|
| random split | ids interleaved or shuffled; train and test indistinguishable |
| future window | a time column, or an ordered id with drift that continues into test |
| held-out entities | repeated entities in train, entity ids absent from test |

In [ ]:
datelike = [
    c
    for c in train.columns
    if pd.api.types.is_datetime64_any_dtype(train[c]) or re.search(r"date|timestamp|_dt$|_ts$", c)
]
print("date/time-like columns:", datelike)
print(f"\n{ID} unique in train: {train[ID].is_unique}   in test: {test[ID].is_unique}")
print(
    f"train {ID}: [{train[ID].min()}, {train[ID].max()}]   contiguous: "
    f"{(np.sort(train[ID].values) == np.arange(train[ID].min(), train[ID].max() + 1)).all()}"
)
print(
    f"test  {ID}: [{test[ID].min()}, {test[ID].max()}]   contiguous: "
    f"{(np.sort(test[ID].values) == np.arange(test[ID].min(), test[ID].max() + 1)).all()}"
)
print(f"id overlap: {len(set(train[ID]) & set(test[ID]))}")
print(f"gap test.min - train.max: {test[ID].min() - train[ID].max()}")

No time column exists. `id` is a contiguous counter and test simply continues it
— so **id order is the only thing that could encode time**, and section 2.2 tests
whether it does.

### 2.1 Is there an entity to hold out?

There is no `customerid` column. The nearest thing to an entity is a repeated
feature vector — the same customer appearing twice.

In [ ]:
dups = train.duplicated(subset=features).sum()
print(f"duplicate feature vectors in train: {dups} ({dups / len(train):.4%})")
print(f"unique feature vectors: {train[features].drop_duplicates().shape[0]:,} of {len(train):,}")

# exact-match overlap between the two frames, on a sample for speed
key_tr = set(map(tuple, train[features].astype(str).to_numpy()))
sample = test[features].astype(str).to_numpy()[:50_000]
overlap = float(np.mean([tuple(r) in key_tr for r in sample]))
print(f"test rows whose exact feature vector also appears in train: {overlap:.4%}")

**COLD_FRACTION is not applicable.** Every row is a distinct feature vector; no
entity repeats within train and none is shared with test. There is no entity to
be cold, so `group` and `time_coldstart` are both ruled out — there is nothing
to group by.

### 2.2 Does id order encode time?

If the organisers cut the test set off the end of a time-ordered file, the churn
rate and the covariates would drift along `id` *and the drift would continue
into test*. Measure the trend inside train, then check whether test looks like
its continuation.

In [ ]:
bins = pd.qcut(train[ID], 40, labels=False)
by_bin = pd.DataFrame(
    {
        "churn_rate": y.groupby(bins).mean(),
        "tenure": train["tenure"].groupby(bins).mean(),
        "monthlycharges": train["monthlycharges"].groupby(bins).mean(),
    }
)

print(f"corr(id, churn)  train: {np.corrcoef(train[ID], y)[0, 1]:+.5f}")
print(f"corr(id, tenure) train: {np.corrcoef(train[ID], train['tenure'])[0, 1]:+.5f}")
print(f"corr(id, tenure) test:  {np.corrcoef(test[ID], test['tenure'])[0, 1]:+.5f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, col, ref in zip(
    axes, by_bin.columns, [None, test["tenure"].mean(), test["monthlycharges"].mean()]
):
    ax.plot(by_bin.index, by_bin[col], marker="o", ms=3)
    if ref is not None:
        ax.axhline(ref, color="crimson", ls="--", lw=1, label="test mean")
        ax.legend()
    ax.set_title(f"{col} by id ventile")
    ax.set_xlabel("id bin (40 quantiles)")
fig.suptitle("Drift along id order — flat after the first few bins")
fig.tight_layout()
savefig(fig, "01_drift_along_id")
by_bin.head(5).join(by_bin.tail(3), how="outer", rsuffix="_")

The first two or three bins sit a little above the rest (churn ~0.25 vs ~0.22),
after which the series is flat. The correlations are ~0.01–0.02 — statistically
detectable at n=594k and practically nothing — and the tenure trend **does not
continue into test**: `corr(id, tenure)` is +0.019 in train and +0.005 in test,
i.e. it disappears rather than extending. That is the signature of a mild
artefact at the head of the file, not of a temporal ordering.

### 2.3 Adversarial validation — the decisive test

Fit a classifier to separate train rows from test rows on the features alone. If
the split is random the two are exchangeable and the AUC sits at 0.5. Anything
much above that means test was selected by *something*, and that something has
to be reproduced in the folds.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

X_adv = pd.concat([train[features], test[features]], ignore_index=True)
for c in cat_cols:
    X_adv[c] = X_adv[c].astype("category")
y_adv = np.r_[np.zeros(len(train)), np.ones(len(test))]

oof_adv = np.zeros(len(X_adv))
for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=C.SEED).split(X_adv, y_adv):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1)
    m.fit(X_adv.iloc[tr_i], y_adv[tr_i])
    oof_adv[va_i] = m.predict_proba(X_adv.iloc[va_i])[:, 1]

adv_auc = roc_auc_score(y_adv, oof_adv)
print(f"adversarial AUC (train vs test): {adv_auc:.5f}")

In [ ]:
drift = []
for c in cat_cols:
    a = train[c].value_counts(normalize=True)
    b = test[c].value_counts(normalize=True)
    drift.append({"column": c, "max_abs_share_diff": (a - b).abs().max()})
for c in num_cols:
    drift.append(
        {
            "column": c,
            "train_mean": train[c].mean(),
            "test_mean": test[c].mean(),
            "std_diff_in_sds": (test[c].mean() - train[c].mean()) / train[c].std(),
        }
    )
pd.DataFrame(drift).set_index("column")

**Adversarial AUC = 0.511.** With ~849k rows the standard error is ~0.0007, so
this is measurably above 0.5 but a model that can barely tell the two frames
apart is, for CV purposes, one that cannot. No categorical marginal differs by
more than 0.8 percentage points; every numeric mean is within 0.03 SD.

Train and test are drawn from the same distribution. **The split is random.**

## 3. The decision

> The test set is a random 30% slice of one synthetic table: one row per
> customer, no time column, no repeated entity, no distribution shift.

Therefore `CV_SCHEME = "stratified"`, `N_FOLDS = 5` — unchanged from the
placeholder in `config.py`, but now backed by measurement rather than by
default. `GROUP_COL` and `TIME_COL` stay empty; `COLD_FRACTION` is not
applicable.

Stratified rather than plain `kfold` because the positive rate is 22.5% — at
5 folds the fold-to-fold variation in class balance is small either way, but
stratifying costs nothing and removes it as a source of noise.

Consequences to record:

- The folds **partition the rows** — every training row is validated exactly
  once, so the OOF array is complete and `cv.add_fold_column` is valid.
- OOF and test come from the same distribution, so CV should track the public
  LB closely. A CV-LB gap here would indicate a bug, not a distribution shift.
- Sampling noise at ~119k validation rows and AUC ≈ 0.91 is about ±0.0012 per
  fold and ±0.0005 on the pooled OOF. Every run is scored on identical folds,
  so paired comparisons cancel part of that: judge a change by whether the
  per-fold differences agree in sign, not against one fixed threshold.

In [ ]:
folds_are_valid = (
    train[ID].is_unique
    and train[features].duplicated().sum() == 0
    and not any(pd.api.types.is_datetime64_any_dtype(train[c]) for c in train.columns)
    and adv_auc < 0.55
)
print(f"stratified folds justified: {folds_are_valid}")
print(
    f"config.py -> CV_SCHEME={C.CV_SCHEME!r}  N_FOLDS={C.N_FOLDS}  "
    f"GROUP_COL={C.GROUP_COL!r}  TIME_COL={C.TIME_COL!r}"
)

from src.cv import add_fold_column, fold_fingerprint

folded = add_fold_column(train)
print("\nrows and positive rate per fold:")
print(folded.groupby("fold")[TARGET].agg(n="size", pos_rate="mean"))
print(f"\nfold fingerprint: {fold_fingerprint(train)}")

## 4. Name and dtype mismatches — found here, fixed in `src/`

Three of them sat between this notebook and a working submission. All are fixed;
recorded so the reasoning survives.

1. **`config.TARGET` was `"Churn"`, which never matches a column.**
   `basic_clean` lower-cases the headers, so `train[C.TARGET]` in `train.py`
   raised `KeyError`. Now `"churn"`.
2. **The label was `"Yes"`/`"No"`, not `1`/`0`.** `train.py` passed `y_raw`
   straight to LightGBM and to `roc_auc_score`, neither of which accepts text.
   `data.encode_target` now maps it to int8, asserting that no value falls
   outside `TARGET_MAP`, and runs after `reduce_memory` so the int8 survives.
3. **The submission column keeps the competition's own casing.**
   `load_sample_submission` does not call `basic_clean`, so the frame has
   `Churn` while `C.TARGET` is `churn`. `sub[C.TARGET] = test_pred` *added* a
   second column and left the real one all-zeros — a file that uploads cleanly
   and scores 0.5. `train.py` now writes whichever column is not `ID_COL`.

## 5. Feature ideas for the baseline (not yet built)

- `totalcharges / (tenure * monthlycharges)` — the residual against the implied
  bill. It is centred on 1.0 with a long tail, so it encodes plan changes and
  one-off charges that neither input carries alone.
- `totalcharges / tenure` — realised average monthly spend vs. current rate.
- Count of subscribed add-ons (the six `"Yes"`/`"No"`/`"No internet service"`
  columns collapsed to a 0–6 integer).
- `tenure` bucketed at the contract boundaries (12, 24 months), where the
  hazard visibly steps.

Nothing here needs target encoding: 2–4 levels per categorical, no unseen
levels, native categorical support in both boosters.